# 🚀 A Tour of the Gemini API — Hands-on Workshop

This notebook walks you through the key capabilities of the **Gemini API** as covered in the presentation.  
Each section is self-contained — run the cells top-to-bottom within each section.

**Sections:**
1. Setup & Installation
2. Basic Text Generation (Gemini 3)
3. Tools & Agents
   - Google Search Grounding
   - URL Context
   - Google Maps Grounding
   - Code Execution
   - Function Calling
   - File Search (RAG)
4. GenMedia — Image Generation (Nano Banana)
5. Audio & Speech (TTS)
6. Live API (Streaming)

---

## 1. Setup & Installation

### Get your API key
1. Visit [Google AI Studio](https://aistudio.google.com)
2. Click **Get API key**
3. Copy it and paste below

In [ ]:
# Install the Google GenAI SDK
!pip install -q google-genai Pillow

In [ ]:
import os
from google import genai
from google.genai import types

# ============================
# 👇 PASTE YOUR API KEY HERE
# ============================
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

# You can also set it via environment variable:
# export GOOGLE_API_KEY="your-key"
# GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

client = genai.Client(api_key=GOOGLE_API_KEY)
print("✅ Client initialized successfully!")

---
## 2. Basic Text Generation — Gemini 3

Gemini 3 represents the shift to an **era of action**:  
- Gemini 1 → Understanding  
- Gemini 2 → Thinking  
- Gemini 3 → **Action**

Available models:
- `gemini-3-pro-preview` — Most capable
- `gemini-3-flash-preview` — Fast & efficient

In [ ]:
# --- Simple text generation with Gemini 3 Flash ---

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="How does AI work? Explain in 3 sentences."
)
print(response.text)

In [ ]:
# --- Gemini 3 Pro — More powerful reasoning ---

response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents="What are the implications of quantum computing on current encryption standards?"
)
print(response.text)

### 2.1 New API Features — Thinking Level

Control the model's internal reasoning with `thinking_level`: `high` or `low`.

In [ ]:
# --- Thinking level control ---

# High thinking — deeper reasoning
response_high = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Solve step-by-step: If a train leaves at 9am going 60mph, and another at 10am going 90mph, when does the second catch up?",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="HIGH")
    )
)

# Print thinking and response
for part in response_high.candidates[0].content.parts:
    if part.thought:
        print("💭 THINKING:", part.text[:300], "...\n")
    else:
        print("📝 RESPONSE:", part.text)

---
## 3. Tools & Agents

Gemini models support powerful tool integrations:
- **Google Search** — Real-time web grounding
- **URL Context** — Extract content from web pages
- **Google Maps** — Location-aware responses
- **Code Execution** — Run Python in the model
- **Function Calling** — Connect to your own APIs
- **File Search** — RAG over your documents

### 3.1 Google Search Grounding

Connect Gemini to real-time web content for factual, up-to-date answers with citations.

In [ ]:
# --- Google Search grounding ---

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="What are the latest AI news this week?",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )
)

print(response.text)

# Print grounding metadata if available
if response.candidates[0].grounding_metadata:
    metadata = response.candidates[0].grounding_metadata
    if metadata.search_entry_point:
        print("\n🔗 Search query:", metadata.search_entry_point.rendered_content[:200] if metadata.search_entry_point.rendered_content else "N/A")

### 3.2 URL Context

Natively extract and process content from user-provided web page URLs.

In [ ]:
# --- URL Context: Extract & compare web content ---

url1 = "https://blog.google/technology/ai/"

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=f"Summarize the latest posts from this blog: {url1}",
    config=types.GenerateContentConfig(
        tools=[types.Tool(url_context=types.UrlContext())]
    )
)

print(response.text)

In [ ]:
# --- URL Context: Compare two web pages ---

url1 = "https://en.wikipedia.org/wiki/Python_(programming_language)"
url2 = "https://en.wikipedia.org/wiki/Rust_(programming_language)"

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=f"Compare the key features and use cases of Python vs Rust based on these pages: {url1} and {url2}",
    config=types.GenerateContentConfig(
        tools=[types.Tool(url_context=types.UrlContext())]
    )
)

print(response.text)

### 3.3 Google Maps Grounding

Connect Gemini with Google Maps for location-aware, factually accurate responses.

In [ ]:
# --- Google Maps grounding ---

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What are the best Italian restaurants within a 15-minute walk from here?",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_maps=types.GoogleMaps())],
        tool_config=types.ToolConfig(
            retrieval_config=types.RetrievalConfig(
                lat_lng=types.LatLng(
                    latitude=21.0285,   # Hanoi — change to your location!
                    longitude=105.8542
                )
            )
        ),
    )
)

print(response.text)

### 3.4 Code Execution

Enable the model to generate **and run** Python code server-side.

In [ ]:
# --- Code Execution: Let Gemini write & run Python ---

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="What is the sum of the first 50 prime numbers? Write and run the code.",
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution())]
    )
)

# Display all parts (code + output + text)
for part in response.candidates[0].content.parts:
    if part.executable_code:
        print("💻 CODE:")
        print(part.executable_code.code)
        print()
    elif part.code_execution_result:
        print("📊 OUTPUT:")
        print(part.code_execution_result.output)
        print()
    elif part.text:
        print("📝 TEXT:")
        print(part.text)
        print()

In [ ]:
# --- Code Execution: Data analysis ---

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="""Generate sample sales data for 12 months, then:
1. Calculate monthly growth rates
2. Find the best and worst months
3. Calculate the average monthly revenue
Show the code and results.""",
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution())]
    )
)

for part in response.candidates[0].content.parts:
    if part.executable_code:
        print("💻 CODE:")
        print(part.executable_code.code)
        print()
    elif part.code_execution_result:
        print("📊 OUTPUT:")
        print(part.code_execution_result.output)
        print()
    elif part.text:
        print("📝", part.text)

### 3.5 Function Calling

Function calling transforms LLMs into systems that interact with external tools and APIs.  
You define functions, and the model decides when and how to call them.

In [ ]:
# --- Function Calling: Weather example ---

# Step 1: Define the function declaration (schema for the model)
get_weather_func = types.FunctionDeclaration(
    name="get_weather",
    description="Get the current weather for a given city",
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "city": types.Schema(type="STRING", description="The city name"),
            "unit": types.Schema(type="STRING", enum=["celsius", "fahrenheit"], description="Temperature unit"),
        },
        required=["city"],
    ),
)

# Step 2: Send user query with the tool
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="What's the weather like in Hanoi today?",
    config=types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=[get_weather_func])]
    )
)

# Step 3: The model returns a function call (not the result!)
function_call = response.candidates[0].content.parts[0].function_call
print(f"🔧 Model wants to call: {function_call.name}")
print(f"   With args: {dict(function_call.args)}")

In [ ]:
# Step 4: Simulate the function execution & send result back

# In a real app, you'd call your actual weather API here
fake_weather_result = {"temperature": 28, "condition": "Partly cloudy", "humidity": 75}

# Build the conversation with the function result
followup_response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        types.Content(role="user", parts=[types.Part.from_text(text="What's the weather like in Hanoi today?")]),
        types.Content(role="model", parts=[types.Part.from_function_call(
            name="get_weather",
            args={"city": "Hanoi", "unit": "celsius"}
        )]),
        types.Content(role="user", parts=[types.Part.from_function_response(
            name="get_weather",
            response=fake_weather_result
        )]),
    ],
    config=types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=[get_weather_func])]
    )
)

print("🤖", followup_response.text)

### 3.6 File Search (RAG)

Upload documents and let Gemini retrieve relevant chunks to answer questions.

In [ ]:
# --- File Search: Upload a file and ask questions about it ---
# NOTE: This creates server-side resources. Clean up after!

import tempfile

# Create a sample document
sample_text = """Obello Design Inc. AI Product Roadmap 2026

AI Resize: Automatically resize designs for multiple platforms.
Phase 1 focuses on simple layouts. Phase 2 handles complex multi-element designs.

Design Copilot: AI assistant that suggests design improvements in real-time.
Uses a combination of CNN-based evaluation and LLM reasoning.

AI Remix: Transform existing designs into new styles while preserving brand identity.
Leverages few-shot prompting and style transfer techniques.

Brand Voice: Ensures all generated content matches the brand's tone and style.
Built on fine-tuned language models with brand-specific training data.
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write(sample_text)
    temp_path = f.name

# Create a file search store
file_search_store = client.file_search_stores.create(
    config={'display_name': 'workshop-demo-store'}
)
print(f"✅ Created store: {file_search_store.name}")

# Upload the file
operation = client.file_search_stores.upload_to_file_search_store(
    file=temp_path,
    file_search_store_name=file_search_store.name,
    config={'display_name': 'product-roadmap.txt'}
)
print(f"✅ File uploaded")

In [ ]:
# Query the uploaded document
import time
time.sleep(5)  # Wait for indexing

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="What AI products are on the roadmap and what technologies do they use?",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[file_search_store.name]
                )
            )
        ]
    )
)

print(response.text)

In [ ]:
# Cleanup: Delete the file search store
try:
    client.file_search_stores.delete(name=file_search_store.name)
    print("🗑️ Store deleted")
except Exception as e:
    print(f"Cleanup note: {e}")

os.unlink(temp_path)
print("🗑️ Temp file deleted")

---
## 4. GenMedia — Image Generation

### Nano Banana Pro (in Gemini 3 Pro)
- State-of-the-art text rendering in images
- Precision controls & reference images
- Best for complex, sophisticated outputs

### Nano Banana (in Gemini 2.5 Flash) 
- Fast, conversational editing (10-15s)
- Character consistency, local edits

In [ ]:
# --- Image Generation with Nano Banana (Gemini 2.5 Flash) ---

from IPython.display import display, Image as IPImage
from PIL import Image
import io
import base64

response = client.models.generate_content(
    model="gemini-2.5-flash-preview-04-17",
    contents="Generate an image of a cute robot mascot for an AI design startup, in a modern flat illustration style with vibrant colors.",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"]
    )
)

# Display results
for part in response.candidates[0].content.parts:
    if part.text:
        print(part.text)
    elif part.inline_data:
        img_data = part.inline_data.data
        img = Image.open(io.BytesIO(img_data))
        img.save("generated_image.png")
        display(IPImage(data=img_data))
        print("\n✅ Image saved to generated_image.png")

In [ ]:
# --- Nano Banana Pro: Advanced text rendering ---

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents="""Create a professional event badge design with:
- Title: "AI Summit Vietnam 2026"
- Subtitle: "Building Responsible AI"
- Clean modern design with blue and white colors""",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"]
    )
)

for part in response.candidates[0].content.parts:
    if part.text:
        print(part.text)
    elif part.inline_data:
        img_data = part.inline_data.data
        img = Image.open(io.BytesIO(img_data))
        img.save("badge_design.png")
        display(IPImage(data=img_data))
        print("\n✅ Image saved to badge_design.png")

### Imagen 4 — Standalone Image Generation

In [ ]:
# --- Imagen 4: High-quality image generation ---

response = client.models.generate_images(
    model="imagen-4.0-generate-preview-05-20",
    prompt="A serene Japanese zen garden with cherry blossoms, koi pond, and stone lantern, golden hour lighting, photorealistic",
    config=types.GenerateImagesConfig(
        number_of_images=1,
    )
)

for image in response.generated_images:
    img = Image.open(io.BytesIO(image.image.image_bytes))
    img.save("imagen4_output.png")
    display(IPImage(data=image.image.image_bytes))
    print("✅ Imagen 4 image saved")

---
## 5. Audio & Speech

### Text-to-Speech (TTS)
- 2 TTS models: Pro and Flash
- Controllable emotion and style
- Multi-voice conversations
- Ideal for podcasts, audiobooks, customer support

In [ ]:
# --- Text-to-Speech with Gemini ---

import wave
from IPython.display import Audio

response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents="Welcome to the Gemini API workshop! Today we'll explore the amazing capabilities of Google's AI models.",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                    voice_name="Kore"  # Try: Puck, Charon, Kore, Fenrir, Aoede, Leda, Orus, Zephyr
                )
            )
        ),
    ),
)

# Save and play the audio
audio_data = response.candidates[0].content.parts[0].inline_data.data

with wave.open("tts_output.wav", "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(24000)
    wf.writeframes(audio_data)

print("✅ Audio saved to tts_output.wav")
Audio("tts_output.wav")

In [ ]:
# --- Multi-voice TTS: Podcast-style dialog ---

response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents="""Generate a short podcast dialog between two speakers:

Speaker 1 (Kore): Hi everyone, welcome to AI Weekly! I'm excited about today's topic.
Speaker 2 (Puck): Me too! We're covering the latest Gemini 3 release from Google.
Speaker 1 (Kore): That's right. The big theme is shifting from thinking to action.
Speaker 2 (Puck): And the benchmarks are really impressive. Let's dive in!""",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                speaker_voice_configs=[
                    types.SpeakerVoiceConfig(speaker="Speaker 1", voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
                    )),
                    types.SpeakerVoiceConfig(speaker="Speaker 2", voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
                    )),
                ]
            )
        ),
    ),
)

audio_data = response.candidates[0].content.parts[0].inline_data.data

with wave.open("podcast_output.wav", "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(24000)
    wf.writeframes(audio_data)

print("✅ Podcast audio saved to podcast_output.wav")
Audio("podcast_output.wav")

---
## 6. Live API (Streaming)

The Live API enables real-time, bidirectional streaming of audio and video.  
Key features:
- Voice-activity detection (automatic, configurable, disabled)
- Session management (compression, resumption)
- Tool chaining
- Native audio output with affective dialog

> ⚠️ The Live API uses WebSockets and is best suited for standalone scripts or web apps,  
> not Jupyter notebooks. Below is a minimal example you can adapt.

In [ ]:
# --- Live API: Minimal text-based streaming example ---
# (Audio streaming requires microphone access — see standalone script below)

import asyncio

async def live_api_text_demo():
    """Simple Live API demo with text input/output."""
    model = "gemini-2.5-flash-native-audio-preview-12-2025"
    config = {
        "response_modalities": ["TEXT"],
        "system_instruction": "You are a helpful assistant. Keep responses brief.",
    }

    async with client.aio.live.connect(model=model, config=config) as session:
        # Send a text message
        await session.send_client_content(
            turns=types.Content(role="user", parts=[types.Part.from_text(text="Tell me a fun fact about Vietnam in one sentence.")])
        )

        # Receive the response
        full_response = ""
        async for response in session.receive():
            if response.server_content and response.server_content.model_turn:
                for part in response.server_content.model_turn.parts:
                    if part.text:
                        full_response += part.text
            if response.server_content and response.server_content.turn_complete:
                break

        print("🤖", full_response)

# Run the async function
await live_api_text_demo()

In [ ]:
# --- Live API: Full audio streaming script (save and run separately) ---

live_api_script = '''
"""Live API Audio Streaming Demo

Run this as a standalone script (not in Jupyter):
    pip install google-genai pyaudio
    python live_api_audio.py
"""
import asyncio
import pyaudio
from google import genai
from google.genai import types

GOOGLE_API_KEY = "YOUR_API_KEY_HERE"
client = genai.Client(api_key=GOOGLE_API_KEY)

FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
CHUNK = 1024

async def main():
    model = "gemini-2.5-flash-native-audio-preview-12-2025"
    config = {
        "response_modalities": ["AUDIO"],
        "system_instruction": "You are a helpful assistant.",
        "tools": [{"google_search": {}}],
    }

    p = pyaudio.PyAudio()

    async with client.aio.live.connect(model=model, config=config) as session:
        # Input stream (microphone)
        input_stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE,
                              input=True, frames_per_buffer=CHUNK)
        # Output stream (speaker)
        output_stream = p.open(format=FORMAT, channels=CHANNELS, rate=24000,
                               output=True, frames_per_buffer=CHUNK)

        print("🎤 Speak now... (Ctrl+C to stop)")

        async def send_audio():
            while True:
                audio_bytes = input_stream.read(CHUNK, exception_on_overflow=False)
                await session.send_realtime_input(
                    audio=types.Blob(data=audio_bytes, mime_type="audio/pcm;rate=16000")
                )
                await asyncio.sleep(0.01)

        async def receive_audio():
            async for response in session.receive():
                if response.data:
                    output_stream.write(response.data)

        await asyncio.gather(send_audio(), receive_audio())

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("live_api_audio.py", "w") as f:
    f.write(live_api_script)

print("✅ Saved live_api_audio.py — run it outside Jupyter with: python live_api_audio.py")

---
## 7. Bonus: Agent Frameworks

Gemini works with all major agent frameworks. Here are quick-start snippets.

In [ ]:
# --- Google ADK (Agent Development Kit) ---
# pip install google-adk

adk_example = '''
from google.adk.agents import Agent
from google.adk.tools import google_search

agent = Agent(
    name="search_assistant",
    model="gemini-3-flash-preview",
    instruction="""You are a helpful assistant. Answer user
                   questions using Google Search when needed.""",
    tools=[google_search]
)
'''
print("🔧 Google ADK Example:")
print(adk_example)

In [ ]:
# --- LangChain / LangGraph ---
# pip install langchain-google-genai

langchain_example = '''
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Use with LangGraph for agent workflows
# graph = builder.compile(name="ReAct Agent")
'''
print("🦜 LangChain Example:")
print(langchain_example)

In [ ]:
# --- CrewAI ---
# pip install crewai

crewai_example = '''
from crewai import LLM, Crew, Process

llm = LLM(
    model="gemini/gemini-3-flash-preview",
    temperature=1.0,
)

# Define crew with agents, tasks, and process
support_analysis_crew = Crew(
    agents=[data_analyst, process_optimizer, report_writer],
    tasks=[analysis_task, optimization_task, report_task],
    process=Process.sequential,
    verbose=True
)
'''
print("🚀 CrewAI Example:")
print(crewai_example)

In [ ]:
# --- LlamaIndex ---
# pip install llama-index-llms-google-genai

llamaindex_example = '''
from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(model="gemini-3-flash-preview")

# Multi-agent workflow
agent_workflow = AgentWorkflow(
    agents=[research_agent, write_agent, review_agent],
    root_agent=research_agent.name,
    initial_state={
        "research_notes": {},
        "report_content": "Not written yet.",
        "review": "Review required.",
    },
)
'''
print("🦙 LlamaIndex Example:")
print(llamaindex_example)

---
## 📚 Resources

| Resource | Link |
|----------|------|
| Google AI Studio | [aistudio.google.com](https://aistudio.google.com) |
| Gemini API Docs | [ai.google.dev/gemini-api](https://ai.google.dev/gemini-api) |
| Cookbook (code examples) | [goo.gle/cookbook](https://goo.gle/cookbook) |
| ADK Docs | [google.github.io/adk-docs](https://google.github.io/adk-docs/) |
| ADK Streaming | [google.github.io/adk-docs/streaming](https://google.github.io/adk-docs/streaming/) |
| Lyria RealTime | [goo.gle/lyria-realtime](https://goo.gle/lyria-realtime) |

---

**Happy building!** 🚀